# Your AI Adoption Lift Is a Selection Effect

Companion notebook for the Towards Data Science article. Everything here runs end to end on a synthetic dataset and reproduces every number and figure in the text.

**What this notebook does**

1. Simulates 40,000 B2B accounts where an AI assistant is gated at 25 seats, adoption is voluntary, and a latent *engagement* variable drives both adoption and retention.
2. Shows the naive adopter gap (+15.4 pp) and the regression-adjusted gap (+13.8 pp) against a true effect of +4 pp.
3. Recovers the effect with a fuzzy regression discontinuity at the eligibility threshold, estimated by 2SLS.
4. Runs the diagnostics: bandwidth sensitivity, discrete running variable, placebo cutoffs, covariate smoothness, and density around the cutoff.
5. Produces both figures from the article.

Run all cells in order. The only non-standard dependency is `linearmodels`, installed in the first cell.

In [ ]:
%pip install -q linearmodels

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from linearmodels.iv import IV2SLS
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})

## 1. The data-generating process

The true effect of adoption is **+4 pp** of 6-month retention. Retention also trends smoothly upward with account size. `engagement` is the confounder: it drives adoption *and* retention, and the analyst never observes it.

In [ ]:
RNG = np.random.default_rng(2026)
N = 40_000

TRUE_EFFECT = 0.04      # +4 pp 6-month retention from adopting the AI assistant
CUTOFF = 25             # AI assistant only available to accounts with >= 25 seats

# Latent engagement: drives adoption AND retention. The analyst never sees it.
engagement = np.clip(RNG.normal(0, 1, N), -2.5, 2.5)

# Seats: skewed, integer, mildly correlated with engagement.
seats = np.clip(
    np.round(np.exp(RNG.normal(3.2, 0.6, N) + 0.10 * engagement)), 3, 300
).astype(int)
tenure = RNG.uniform(6, 36, N)

eligible = (seats >= CUTOFF).astype(int)

# Adoption is voluntary among eligible accounts; engaged accounts opt in more.
p_adopt = 1 / (1 + np.exp(-(-0.6 + 1.4 * engagement)))
adopted = ((eligible == 1) & (RNG.uniform(size=N) < p_adopt)).astype(int)

# Retention: baseline + engagement + smooth seat trend + the true effect.
p_retain = (0.55
            + 0.10 * engagement
            + 0.03 * (np.log(seats) - np.log(CUTOFF))
            + TRUE_EFFECT * adopted)
retained = (RNG.uniform(size=N) < p_retain).astype(int)

df = pd.DataFrame(dict(seats=seats, tenure=tenure, eligible=eligible,
                       adopted=adopted, retained=retained, engagement=engagement))
df.head()

## 2. Method 1: the slide

Adopters vs. non-adopters. This is what the naive comparison estimates.

In [ ]:
naive = df.groupby('adopted')['retained'].mean()
print(naive.round(3))
print(f'\nNaive adopter gap: {naive[1] - naive[0]:+.3f}')

elig = df[df.eligible == 1]
naive_elig = elig.groupby('adopted')['retained'].mean()
print(f'Adopter gap among eligible accounts only: {naive_elig[1] - naive_elig[0]:+.3f}')

## 3. Method 2: regression adjustment on observables

Controlling for seats and tenure barely moves the number. The oracle model, which conditions on the unobserved `engagement`, recovers the truth, but the analyst does not have that column.

In [ ]:
adj = smf.ols('retained ~ adopted + np.log(seats) + tenure', data=elig).fit(cov_type='HC3')
print(f'Adjusted (observables): {adj.params["adopted"]:+.3f}  (95% CI ±{1.96 * adj.bse["adopted"]:.3f})')

oracle = smf.ols('retained ~ adopted + np.log(seats) + tenure + engagement', data=elig).fit(cov_type='HC3')
print(f'Oracle (with engagement): {oracle.params["adopted"]:+.3f}')

## 4. Method 3: fuzzy regression discontinuity at the 25-seat threshold

Eligibility is the instrument, adoption is the treatment, and 2SLS is the estimator. The function returns the first stage (jump in adoption), the reduced form (jump in retention), and the 2SLS estimate of the adoption effect with a heteroskedasticity-robust standard error.

In [ ]:
def fuzzy_rd(df, cutoff=CUTOFF, bw=10):
    # Keep accounts within `bw` seats of the cutoff on either side.
    w = df[(df.seats >= cutoff - bw) & (df.seats < cutoff + bw)].copy()
    w['x'] = w.seats - cutoff                 # running variable, centred at 0
    w['above'] = (w.x >= 0).astype(int)       # eligibility: the instrument
    w['above_x'] = w.above * w.x              # lets the slope differ by side

    first = smf.ols('adopted ~ above * x', data=w).fit(cov_type='HC1')
    reduced = smf.ols('retained ~ above * x', data=w).fit(cov_type='HC1')

    # Second stage: retention on adoption, with local linear trend each side.
    # First stage (in brackets): adoption instrumented by eligibility.
    model = IV2SLS.from_formula(
        'retained ~ 1 + x + above_x + [adopted ~ above]', data=w
    ).fit(cov_type='robust')

    return dict(n=len(w),
                first_stage=first.params['above'],
                reduced_form=reduced.params['above'],
                reduced_form_ci=reduced.conf_int().loc['above'].values,
                late=model.params['adopted'],
                se=model.std_errors['adopted'],
                ci=model.conf_int().loc['adopted'].values,
                model=model)

r = fuzzy_rd(df, bw=10)
print(f'Bandwidth ±10 seats, n = {r["n"]:,}')
print(f'First stage (jump in adoption):   {r["first_stage"]:.3f}')
print(f'Reduced form (jump in retention): {r["reduced_form"]:+.4f}  95% CI [{r["reduced_form_ci"][0]:+.3f}, {r["reduced_form_ci"][1]:+.3f}]')
print(f'2SLS (effect of adoption):        {r["late"]:+.3f}  se {r["se"]:.3f}  95% CI [{r["ci"][0]:+.3f}, {r["ci"][1]:+.3f}]')

### Figure 1: same data, four estimates

In [ ]:
labels = ['Naive\n(adopters vs\nnon-adopters)', 'Regression\n(seats, tenure)',
          'Fuzzy RD / 2SLS\n(seat threshold)', 'Oracle\n(with engagement)', 'True effect\n(simulation)']
vals = [100 * (naive[1] - naive[0]), 100 * adj.params['adopted'], 100 * r['late'],
        100 * oracle.params['adopted'], 100 * TRUE_EFFECT]
cols = ['#d94141', '#d94141', '#4a86c8', '#444444', '#222222']

fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.bar(labels, vals, color=cols, width=0.6)
ax.errorbar([1, 2], [vals[1], vals[2]],
            yerr=[196 * adj.bse['adopted'], 196 * r['se']],
            fmt='none', ecolor='black', capsize=5, lw=1.2)
for i, v in enumerate(vals):
    ax.text(i, v + (8.5 if i == 2 else 0.8), f'{v:+.1f} pp', ha='center', fontweight='bold')
ax.axhline(100 * TRUE_EFFECT, ls='--', color='grey', lw=1)
ax.set_ylabel('Estimated retention lift from adoption (pp)')
ax.set_ylim(0, 20)
ax.set_title('Same data, four estimates of the AI assistant effect', fontweight='bold')
plt.tight_layout()
plt.show()

### Figure 2: the discontinuity

In [ ]:
w = df[(df.seats >= 13) & (df.seats <= 37)]
g = w.groupby('seats').agg(adopt=('adopted', 'mean'), ret=('retained', 'mean'), n=('seats', 'size')).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, col, ycol, title, yl in [
    (axes[0], 'adopt', 'adopted', 'First stage: adoption jumps at 25 seats', 'Adoption rate'),
    (axes[1], 'ret', 'retained', 'Reduced form: retention jumps at 25 seats', '6-month retention rate')]:
    ax.scatter(g.seats, g[col], s=g.n / 12, color='#4a86c8', alpha=.8)
    for mask, xs in [(w.seats < CUTOFF, np.array([13, 24.999])), (w.seats >= CUTOFF, np.array([25, 37]))]:
        m = smf.ols(f'{ycol} ~ seats', data=w[mask]).fit()
        ax.plot(xs, m.params['Intercept'] + m.params['seats'] * xs, color='#d94141', lw=2)
    ax.axvline(24.5, ls='--', color='grey')
    ax.set_xlabel('Seats on the account'); ax.set_ylabel(yl)
    ax.set_title(title, fontweight='bold', fontsize=11)
axes[0].set_ylim(-0.03, 0.5); axes[1].set_ylim(0.50, 0.62)
plt.tight_layout()
plt.show()

## 5. Diagnostics

### 5.1 Bandwidth sensitivity

Narrow is more credible and noisier; wide is more precise and starts picking up curvature. Report the range.

In [ ]:
rows = []
for bw in [5, 10, 15, 20]:
    r_bw = fuzzy_rd(df, bw=bw)
    rows.append(dict(bandwidth=f'±{bw}', n=r_bw['n'], first_stage=round(r_bw['first_stage'], 3),
                     estimate_2sls=round(r_bw['late'], 3),
                     ci_low=round(r_bw['ci'][0], 3), ci_high=round(r_bw['ci'][1], 3)))
pd.DataFrame(rows)

### 5.2 Discrete running variable

Seats are integers. Clustering standard errors by the running variable's values (Lee and Card, 2008) is common advice, but it can understate uncertainty (Kolesár and Rothe, 2018). Here it shrinks the SE substantially. The article keeps the robust interval as a measure of sampling uncertainty and treats the bandwidth table as part of the primary result.

In [ ]:
w10 = df[(df.seats >= 15) & (df.seats < 35)].copy()
w10['x'] = w10.seats - CUTOFF; w10['above'] = (w10.x >= 0).astype(int); w10['above_x'] = w10.above * w10.x
m_cl = IV2SLS.from_formula('retained ~ 1 + x + above_x + [adopted ~ above]', data=w10).fit(
    cov_type='clustered', clusters=w10.seats)
print(f'2SLS at ±10, robust SE:            {r["se"]:.3f}')
print(f'2SLS at ±10, clustered by seat SE: {m_cl.std_errors["adopted"]:.3f}')

### 5.3 Placebo cutoffs

Same reduced-form regression at seat counts where nothing happens. The windows must not cross the real cutoff: a placebo at 20 with a ±10 window would span 10 to 29 and contain the actual discontinuity.

In [ ]:
for pc in [15, 35, 45, 55]:
    wp = df[(df.seats >= pc - 10) & (df.seats < pc + 10)].copy()
    wp['x'] = wp.seats - pc; wp['above'] = (wp.x >= 0).astype(int)
    m = smf.ols('retained ~ above * x', data=wp).fit(cov_type='HC1')
    print(f'placebo cutoff {pc}: n={len(wp):6,}  jump={m.params["above"]:+.4f}  se={m.bse["above"]:.4f}')

### 5.4 Covariate smoothness

Pre-treatment covariates should not jump at the threshold. In the simulation we can also check the unobserved confounder itself, which is the point of the exercise.

In [ ]:
for c in ['tenure', 'engagement']:
    m = smf.ols(f'{c} ~ above * x', data=w10).fit(cov_type='HC1')
    print(f'{c:12s} jump at cutoff = {m.params["above"]:+.3f}  (se {m.bse["above"]:.3f})')

### 5.5 Density around the cutoff

A pile-up at exactly 25 seats would mean the running variable is being manipulated (for example, sales upselling 22-seat accounts to unlock the feature). Smooth here; check it in your own data before anything else.

In [ ]:
counts = df.seats.value_counts().sort_index().loc[18:32]
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(counts.index, counts.values, color=['#d94141' if s == CUTOFF else '#4a86c8' for s in counts.index])
ax.axvline(24.5, ls='--', color='grey')
ax.set_xlabel('Seats on the account'); ax.set_ylabel('Accounts')
ax.set_title('Density of the running variable around the cutoff', fontweight='bold')
plt.tight_layout(); plt.show()
print(counts.to_dict())

## 6. Which number goes on the slide

In [ ]:
summary = pd.DataFrame([
    ['Naive adopter gap',            f'{100*(naive[1]-naive[0]):+.1f} pp', 'Ready vs. unready organizations, with a feature flag'],
    ['Regression-adjusted',          f'{100*adj.params["adopted"]:+.1f} pp', 'Same thing, holding seats and tenure fixed'],
    ['RD reduced form at 25 seats',  f'{100*r["reduced_form"]:+.1f} pp ({100*r["reduced_form_ci"][0]:+.1f} to {100*r["reduced_form_ci"][1]:+.1f})', 'Effect of offering access at the eligibility margin'],
    ['RD 2SLS at 25 seats',          f'{100*r["late"]:+.1f} pp ({100*r["ci"][0]:+.1f} to {100*r["ci"][1]:+.1f})', 'Effect of adoption for accounts induced to adopt by eligibility'],
    ['True effect (simulation)',     f'{100*TRUE_EFFECT:+.1f} pp', ''],
], columns=['Estimate', 'Value', 'What it is'])
summary.style.hide(axis='index')

---
*References: Lee, D. S. and Card, D. (2008). Regression discontinuity inference with specification error. Journal of Econometrics. Kolesár, M. and Rothe, C. (2018). Inference in regression discontinuity designs with a discrete running variable. American Economic Review.*